In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

import torch
import transformers

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/connection/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-11-12 18:21:16 __main__ INFO     torch.__version__='2.9.0+cu128', torch.version.cuda='12.8'
2025-11-12 18:21:17 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100-SXM4-80GB'
2025-11-12 18:21:17 __main__ INFO     transformers.__version__='4.57.1'


## Loading the LM

In [3]:
from src.utils.training_utils import get_device_map

# model_key = "meta-llama/Llama-3.2-3B"
# model_key = "meta-llama/Llama-3.1-8B-Instruct"
model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "meta-llama/Llama-3.1-405B-Instruct"

# model_key = "google/gemma-2-9b-it"
# model_key = "google/gemma-3-12b-it"
# model_key = "google/gemma-2-27b-it"

# model_key = "openai/gpt-oss-20b"

# model_key = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# model_key = "allenai/OLMo-2-1124-7B-Instruct"
# model_key = "allenai/OLMo-7B-0424-hf"

# model_key = "Qwen/Qwen2-7B"
# model_key = "Qwen/Qwen2.5-14B-Instruct"
# model_key = "Qwen/Qwen2.5-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "Qwen/Qwen3-1.7B"
# model_key = "Qwen/Qwen3-4B"
# model_key = "Qwen/Qwen3-8B"
# model_key = "Qwen/Qwen3-14B"
# model_key = "Qwen/Qwen3-32B"

# device_map = get_device_map(model_key, 30, n_gpus=8)
# device_map

2025-11-12 18:21:23 git.cmd DEBUG    Popen(['git', 'version'], cwd=/disk/u/arnab/Codes/Projects/filter/notebooks, stdin=None, shell=False, universal_newlines=False)
2025-11-12 18:21:23 git.cmd DEBUG    Popen(['git', 'version'], cwd=/disk/u/arnab/Codes/Projects/filter/notebooks, stdin=None, shell=False, universal_newlines=False)


In [ ]:
from src.models import ModelandTokenizer

# from transformers import BitsAndBytesConfig

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    # device_map=device_map,
    device_map="auto",
    # quantization_config = BitsAndBytesConfig(
    #     # load_in_4bit=True
    #     load_in_8bit=True
    # )
    attn_implementation="eager",
)

2025-11-12 18:21:29 src.models WARNING  meta-llama/Llama-3.3-70B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2025-11-12 18:21:29 urllib3.connectionpool DEBUG    Starting new HTTPS connection (1): huggingface.co:443


2025-11-12 18:21:29 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.3-70B-Instruct/resolve/main/config.json HTTP/1.1" 200 0
2025-11-12 18:21:30 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.3-70B-Instruct/resolve/main/tokenizer_config.json HTTP/1.1" 200 0
2025-11-12 18:21:30 urllib3.connectionpool DEBUG    https://huggingface.co:443 "GET /api/models/meta-llama/Llama-3.3-70B-Instruct/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64


Loading checkpoint shards:  17%|█▋        | 5/30 [00:03<00:19,  1.29it/s]

## Saving the selection data
> For baseline evaluation. So that every LM is evaluated on the same set of data.

In [30]:
from src.selection.data import SelectOneTask
from typing import Literal

##########################################################
prompt_template_idx = 3  # try out different templates
option_style: Literal["single_line", "numbered"] = "single_line"
n_distractors = (
    5  # number of distractors. total options = n_distractors + 1 for SingleOne task
)
##########################################################

# symantic_type = "objects"
# symantic_type = "profession"
symantic_type = "nationality"

select_task = SelectOneTask.load(
    path=os.path.join(
        env_utils.DEFAULT_DATA_DIR, "selection", f"{symantic_type}.json"
    )
)
select_task.categories

['name', 'prompt_templates', 'categories']


['United States',
 'United Kingdom',
 'Canada',
 'Australia',
 'India',
 'Brazil',
 'France',
 'Germany',
 'Japan',
 'China',
 'Argentina',
 'Italy']

In [31]:
sample = select_task.get_random_sample(
    mt=mt,
    option_style=option_style,
    prompt_template_idx=prompt_template_idx,
    # category="soccer player",
    filter_by_lm_prediction=True,
)

print(sample.prompt(), ">>", sample.obj)
print(f'"{mt.tokenizer.decode([sample.ans_token_id])}"')

2025-11-12 17:34:45 src.selection.data ERROR    Sample = Stephen King -> Beyoncé (1): ['Keanu Reeves', 'Beyoncé', 'Liam Hemsworth', 'Roberto Carlos', 'Ronaldo Nazário', 'Ed Sheeran']
    Top prediction (1, PredictedToken(token=' K', prob=0.62890625, logit=20.625, token_id=735, metadata=None)) does not match the object Beyoncé[54992, " Bey"].
    Retry count: 1. Retrying ...
    
Options: Brad Pitt, Russell Crowe, Deepika Padukone, Yao Ming, Heidi Klum, Liam Hemsworth.
Who among these people mentioned above is from United States?
Answer: >> Brad Pitt
" Brad"


In [32]:
from tqdm.auto import tqdm

################################################################################################
LIMIT = 1024
N_DISTRACTORS = 5
DS_ROOT = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "selection/baseline", select_task.task_name
)
################################################################################################

os.makedirs(DS_ROOT, exist_ok=True)

evaluation_samples = []
for _ in tqdm(range(LIMIT)):
    sample = select_task.get_random_sample(
        mt=mt,
        option_style=option_style,
        prompt_template_idx=prompt_template_idx,
        n_distractors=N_DISTRACTORS,
        filter_by_lm_prediction=False,
    )
    evaluation_samples.append(sample)

with open(os.path.join(DS_ROOT, f"{symantic_type}.json"), "w") as f:
    json.dump(
        [sample.to_dict() for sample in evaluation_samples],
        f,
        indent=4,
    )

  0%|          | 0/1024 [00:00<?, ?it/s]

100%|██████████| 1024/1024 [00:10<00:00, 96.06it/s]


## Load Evaluation Samples

In [5]:
from src.selection.data import SelectOneTask
from typing import Literal
from src.selection.data import SelectionSample
from src.selection.utils import get_first_token_id
from src.selection.data import MCQify_sample

##########################################################
prompt_template_idx = 3  # try out different templates
option_style: Literal["single_line", "numbered"] = "single_line"
symantic_type = "objects"
# symantic_type = "profession"
# symantic_type = "nationality"
##########################################################

select_task = SelectOneTask.load(
    path=os.path.join(
        env_utils.DEFAULT_DATA_DIR, "selection", f"{symantic_type}.json"
    )
)

DS_ROOT = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "selection/baseline", select_task.task_name
)

with open(os.path.join(DS_ROOT, f"{symantic_type}.json"), "r") as f:
    raw_samples = json.load(f)
evaluation_samples = [SelectionSample.from_dict(d) for d in raw_samples]

prompt_template = select_task.prompt_templates[prompt_template_idx]

for idx in range(len(evaluation_samples)):
    evaluation_samples[idx].prompt_template = prompt_template
    evaluation_samples[idx].option_style = option_style
    evaluation_samples[idx].ans_token_id = get_first_token_id(
        name=evaluation_samples[idx].answer, tokenizer=mt.tokenizer, prefix=" "
    )
    evaluation_samples[idx] = MCQify_sample(
        sample=evaluation_samples[idx], tokenizer=mt.tokenizer
    )

sample = evaluation_samples[12]
print(sample.prompt(), ">>", sample.obj, ">>", f'\"{mt.tokenizer.decode(sample.ans_token_id)}\"')

['name', 'prompt_templates', 'odd_one_prompt_templates', 'order_prompt_templates', 'count_prompt_templates', 'yes_no_prompt_templates', 'first_item_in_cat_prompt_templates', 'last_item_in_cat_prompt_templates', 'categories', 'exclude_categories']
a. Eucalyptus
b. Mirror
c. Comb
d. Socks
e. Birch
f. Pen
Which among these objects mentioned above is a clothing?
Answer: >> Socks >> " d"


In [6]:
from src.selection.utils import get_first_token_id, verify_correct_option
from src.selection.data import get_options_for_answer

result = verify_correct_option(
    mt=mt,
    target=sample.ans_token_id,
    options=get_options_for_answer(sample),
    input=sample.prompt(),
    k=10,
)
result

(True,
 [PredictedToken(token=' d', prob=0.953125, logit=23.5, token_id=499, metadata=None),
  PredictedToken(token=' **', prob=0.036865234375, logit=20.25, token_id=5231, metadata=None),
  PredictedToken(token=' Socks', prob=0.0050048828125, logit=18.25, token_id=114288, metadata=None),
  PredictedToken(token='\n\n', prob=0.0020904541015625, logit=17.375, token_id=109, metadata=None),
  PredictedToken(token='  ', prob=0.00098419189453125, logit=16.625, token_id=139, metadata=None),
  PredictedToken(token='\n', prob=0.000675201416015625, logit=16.25, token_id=108, metadata=None),
  PredictedToken(token=' ', prob=0.00023365020751953125, logit=15.1875, token_id=235248, metadata=None),
  PredictedToken(token='d', prob=0.000133514404296875, logit=14.625, token_id=235258, metadata=None),
  PredictedToken(token=' (', prob=9.1552734375e-05, logit=14.25, token_id=591, metadata=None),
  PredictedToken(token=' socks', prob=7.581710815429688e-05, logit=14.0625, token_id=28931, metadata=None)],
 O

In [7]:
from tqdm import tqdm

results = []
for sample in tqdm(evaluation_samples):
    is_correct, pred, track = verify_correct_option(
        mt=mt,
        target=sample.ans_token_id,
        options=get_options_for_answer(sample),
        input=sample.prompt(),
    )
    results.append(
        {
            "sample": sample,
            "is_correct": is_correct,
            "predicted_option": pred,
            "track": track,
        }
    )

  0%|          | 0/1024 [00:00<?, ?it/s]

100%|██████████| 1024/1024 [02:22<00:00,  7.19it/s]


In [8]:
import numpy as np

ranks = []
logits = []
for result in results:
    sample = result["sample"]
    cur_rank = result["track"][sample.ans_token_id][0]
    ranks.append(cur_rank)
    logits.append(result["track"][sample.ans_token_id][1].logit)

n_correct = sum([1 for result in results if result["is_correct"]])
accuracy = n_correct / len(results)

ranks = np.array(ranks)
ranks_avg = ranks.mean()
ranks_std = ranks.std()

logits = np.array(logits)
logits_avg = logits.mean()
logits_std = logits.std()

print(
    f"Accuracy: {accuracy*100:.2f}% ({n_correct}/{len(results)}) | Avg. Rank: {ranks_avg:.2f} ± {ranks_std:.2f} | Avg. Logit: {logits_avg:.2f} ± {logits_std:.2f}"
)

Accuracy: 97.75% (1001/1024) | Avg. Rank: 1.34 ± 2.91 | Avg. Logit: 22.51 ± 1.37


In [10]:
print(sample.prompt())

a. Tomato
b. Refrigerator
c. Orange
d. Jeans
e. Violin
f. Skis
Which among these objects mentioned above is a sport equipment?
Answer:


In [11]:
failed_cases = [result for result in results if not result["is_correct"]]

In [12]:
sample = failed_cases[5]["sample"]
print(sample.prompt())

a. Lion
b. Chrysanthemum
c. Jacket
d. Bat
e. Cow
f. Daffodil
Which among these objects mentioned above is a sport equipment?
Answer:


In [13]:
sample.answer

'Bat'